# Inception y EfficientNet con PyTorch

Implementaremos un bloque Inception y un bloque MBConv, y probaremos InceptionV3 y EfficientNetB0 con CIFAR-10. Los ajustes cortos son demostrativos, no buscan alcanzar el mejor rendimiento.

## Entorno y muestra de datos

Usa el kernel **IPD434**. CIFAR-10 se descarga si no existe en `../data/cifar10`. Redimensionamos las imágenes a $96\times96$, tamaño compatible con InceptionV3 y más liviano para la demostración.

In [ ]:
from pathlib import Path

import torch
from torch import nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, models, transforms

SEED = 2026
torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_ROOT = Path('../data/cifar10')
transform = transforms.Compose([
    transforms.Resize((96, 96)), transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])
train_full = datasets.CIFAR10(DATA_ROOT, train=True, download=True, transform=transform)
generator = torch.Generator().manual_seed(SEED)
train_set, val_set, unused = random_split(train_full, [128, 32, 49840], generator=generator)
train_loader = DataLoader(train_set, batch_size=8, shuffle=True, num_workers=0)
val_loader = DataLoader(val_set, batch_size=8, shuffle=False, num_workers=0)
print('Dispositivo:', device, '| ejemplos:', len(train_set), len(val_set))

## Bloque Inception

El bloque procesa la misma entrada con filtros de distintos tamaños y concatena sus salidas por canales. Las convoluciones $1\times1$ reducen o mezclan canales antes de operaciones más costosas.

In [ ]:
class InceptionBlock(nn.Module):
    def __init__(self, in_channels, f1, f3_reduce, f3, f5_reduce, f5, pool_proj):
        super().__init__()
        self.branch1 = nn.Sequential(nn.Conv2d(in_channels, f1, 1), nn.ReLU())
        self.branch3 = nn.Sequential(
            nn.Conv2d(in_channels, f3_reduce, 1), nn.ReLU(),
            nn.Conv2d(f3_reduce, f3, 3, padding=1), nn.ReLU(),
        )
        self.branch5 = nn.Sequential(
            nn.Conv2d(in_channels, f5_reduce, 1), nn.ReLU(),
            nn.Conv2d(f5_reduce, f5, 5, padding=2), nn.ReLU(),
        )
        self.branch_pool = nn.Sequential(
            nn.MaxPool2d(3, stride=1, padding=1),
            nn.Conv2d(in_channels, pool_proj, 1), nn.ReLU(),
        )

    def forward(self, x):
        branches = [self.branch1(x), self.branch3(x), self.branch5(x), self.branch_pool(x)]
        return torch.cat(branches, dim=1)

inception = InceptionBlock(192, 64, 96, 128, 16, 32, 32).to(device)
sample = torch.randn(2, 192, 16, 16, device=device)
print('Forma de entrada/salida:', tuple(sample.shape), tuple(inception(sample).shape))

## Bloque MBConv

MBConv expande los canales, aplica una convolución *depthwise*, recalibra canales con Squeeze-and-Excitation y proyecta la salida. Si las formas coinciden, suma un atajo residual.

In [ ]:
class SqueezeExcitation(nn.Module):
    def __init__(self, channels, ratio=4):
        super().__init__()
        hidden = max(1, channels // ratio)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.gates = nn.Sequential(
            nn.Conv2d(channels, hidden, 1), nn.ReLU(),
            nn.Conv2d(hidden, channels, 1), nn.Sigmoid(),
        )

    def forward(self, x):
        return x * self.gates(self.pool(x))

class MBConv(nn.Module):
    def __init__(self, in_channels, out_channels, expansion=6, stride=1):
        super().__init__()
        expanded = in_channels * expansion
        self.expand = nn.Sequential(
            nn.Conv2d(in_channels, expanded, 1, bias=False),
            nn.BatchNorm2d(expanded), nn.SiLU(),
        )
        self.depthwise = nn.Sequential(
            nn.Conv2d(expanded, expanded, 3, stride=stride, padding=1, groups=expanded, bias=False),
            nn.BatchNorm2d(expanded), nn.SiLU(),
        )
        self.se = SqueezeExcitation(expanded)
        self.project = nn.Sequential(
            nn.Conv2d(expanded, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
        )
        self.use_skip = stride == 1 and in_channels == out_channels

    def forward(self, x):
        y = self.project(self.se(self.depthwise(self.expand(x))))
        return x + y if self.use_skip else y

mbconv = MBConv(16, 16).to(device)
print('Salida MBConv:', tuple(mbconv(torch.randn(2, 16, 32, 32, device=device)).shape))

## InceptionV3 y EfficientNetB0

Torchvision ofrece ambas arquitecturas. Las inicializamos sin pesos preentrenados y cambiamos la salida a diez clases. El siguiente paso de entrenamiento es corto y solo verifica el flujo en CIFAR-10.

In [ ]:
criterion = nn.CrossEntropyLoss()
def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)
        correct += (logits.argmax(1) == labels).sum().item()
        total += len(labels)
    return total_loss / total, correct / total

def evaluate(model, loader):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            total_loss += criterion(logits, labels).item() * len(labels)
            correct += (logits.argmax(1) == labels).sum().item()
            total += len(labels)
    return total_loss / total, correct / total

inception_v3 = models.inception_v3(weights=None, aux_logits=False, init_weights=False, num_classes=10).to(device)
inception_optimizer = torch.optim.Adam(inception_v3.parameters(), lr=1e-4)
inception_train_loss, inception_train_acc = train_one_epoch(inception_v3, train_loader, inception_optimizer)
inception_val_loss, inception_val_acc = evaluate(inception_v3, val_loader)
print(f'InceptionV3 | train_loss={inception_train_loss:.3f} | val_acc={inception_val_acc:.3f}')

efficientnet_b0 = models.efficientnet_b0(weights=None, num_classes=10).to(device)
efficientnet_optimizer = torch.optim.Adam(efficientnet_b0.parameters(), lr=1e-4)
efficient_train_loss, efficient_train_acc = train_one_epoch(efficientnet_b0, train_loader, efficientnet_optimizer)
efficient_val_loss, efficient_val_acc = evaluate(efficientnet_b0, val_loader)
print(f'EfficientNetB0 | train_loss={efficient_train_loss:.3f} | val_acc={efficient_val_acc:.3f}')

Un solo paso confirma que las formas, la pérdida y los gradientes funcionan. Para comparar precisión, hay que entrenar con más datos y épocas, fijar validación y reservar prueba.